# Отчет о состоянии проекта: Multimodal RAG для Электроэнергетики

## 1. Что сделано и Зачем (Архитектура Данных)

### А. Данные (Data Mining)
*   **Источник:** Учебный курс по ТОЭ (toehelp.ru), 43 лекции.
*   **Метод:** HTML-парсинг с сохранением структуры (DOM).
*   **Результат:**
    *   `theory.json`: 2000+ текстовых чанков (абзацы теории).
    *   `images.json`: 2500+ изображений (схемы, формулы) в формате PNG.
    *   **Ноу-хау:** Реализован алгоритм "контекстной привязки". Каждая картинка знает ID текста, который шел перед ней. Это критически важно для обучения и оценки мультимодальности.

### Б. Векторные Индексы (Retrieval Backends)

| Тип | Модель | Файл индекса | Научная гипотеза (Зачем?) |
| :--- | :--- | :--- | :--- |
| **Text** | `ruBERT-tiny2` | `idx_text_rubert.bin` | **Baseline.** Быстрая, старая модель. Ожидаем базовое качество. |
| **Text** | `USER-bge-m3` | `idx_text_user_bge.bin` | **SOTA Text.** Современная модель для семантического поиска. Ожидаем прирост качества на сложных вопросах. |
| **Vision** | `OpenAI CLIP` | `idx_clip_openai.bin` | **Baseline Vision.** Модель, обученная на английском. Ожидаем провал на русских запросах ("схема"). |
| **Vision** | `SigLIP` | `idx_siglip.bin` | **Vision SOTA.** Google-модель, лучшая в распознавании деталей (пикселей), но слабее в языке. |
| **Vision** | `Jina CLIP` | `idx_jina.bin` | **Multimodal SOTA.** Специально обучена для RAG и длинных текстов. Ожидаем лидерство. |
| **Full** | `Jina Full` | `idx_jina_full.bin` | **Experimental.** Единый индекс (Текст + Картинки). Проверка гипотезы: может ли одна модель заменить сложный пайплайн. |

### В. Валидация (Synthetic Ground Truth)
*   **Инструмент:** Qwen2.5-VL-7B-Instruct (на A100 GPU).
*   **Результат:** `benchmark_final.json` (430 пар Вопрос-Ответ).
*   **Структура:**
    *   Вопросы по тексту ("Что такое закон Ома?").
    *   Вопросы по схемам ("Как соединены R1 и R2?").
    *   Гибридные вопросы ("Исходя из схемы и описания, определите...").
*   **Метрика успеха:** Поле `gold_ids` содержит правильные ID чанков/картинок.

---
> чтобы сделать оценку поиска по вопросам по схеме искать нужно по answer, не по question

In [1]:
# Установку зависимостей выполняй один раз в окружении сервера:
# pip install -r requirements.txt


In [2]:
import os
import json
import faiss
import torch
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, AutoModel, CLIPProcessor, CLIPModel
BASE_DIR = os.environ.get("EE_RAG_BASE_DIR", os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
INDICES_DIR = os.path.join(BASE_DIR, "indices")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Avoid accidental meta/default-device contexts from external runtimes.
try:
    torch.set_default_device('cpu')
except Exception:
    pass
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"INDICES_DIR: {INDICES_DIR}")
print(f"DEVICE: {DEVICE}")
print("torch default device forced to: cpu")
with open(os.path.join(DATA_DIR, "benchmark_final.json"), 'r') as f:
    benchmark = json.load(f)
print(f"Бенчмарк: {len(benchmark)} вопросов")
def load_json(filename):
    path = os.path.join(INDICES_DIR, filename)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None
mappings = {}
with open(os.path.join(DATA_DIR, "theory.json"), 'r') as f:
    theory_data = json.load(f)
    mappings['text_linear'] = [t['id'] for t in theory_data]
mappings['siglip'] = load_json("idx_siglip_ids.json")
mappings['jina'] = load_json("idx_jina_ids.json")
mappings['clip'] = load_json("idx_clip_openai_ids.json")
mappings['siglip_full'] = load_json("idx_bad_siglip_full_ids.json")
mappings['jina_full'] = load_json("idx_jina_full_ids.json")
print("Маппинги загружены.")


BASE_DIR: /multimodal-rag-article
DATA_DIR: /multimodal-rag-article/data
INDICES_DIR: /multimodal-rag-article/indices
DEVICE: cuda
torch default device forced to: cpu
Бенчмарк: 697 вопросов
Маппинги загружены.


In [3]:
mappings.keys()

dict_keys(['text_linear', 'siglip', 'jina', 'clip', 'siglip_full', 'jina_full'])

Evaluator

In [4]:
class RAGEvaluator:
    def __init__(self, device='cuda'):
        self.device = device
        self.models = {}
        self.indices = {}
        self.error_rows = []
    def reset_error_rows(self):
        self.error_rows = []
    def export_error_rows(self, path):
        import pandas as pd
        pd.DataFrame(self.error_rows).to_csv(path, index=False, encoding='utf-8')
    def load_model(self, model_key):
        if model_key in self.models: return self.models[model_key]
        print(f"Loading model: {model_key}...")
        processor = None
        if model_key == 'rubert':
            model = SentenceTransformer('cointegrated/rubert-tiny2', device=self.device)
        elif model_key == 'bge':
            model = SentenceTransformer('deepvk/USER-bge-m3', device=self.device)
        elif model_key == 'siglip':
            processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
            model = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(self.device)
        elif model_key == 'jina':
            # Jina remote-code model can fail if default device context is 'meta'.
            # Force a concrete default device for loading, disable empty-weight init,
            # then move model to target device.
            processor = AutoProcessor.from_pretrained(
                "jinaai/jina-clip-v1",
                trust_remote_code=True,
                use_fast=False
            )
            try:
                torch.set_default_device('cpu')
            except Exception:
                pass
            load_dtype = torch.float16 if self.device == 'cuda' else torch.float32
            model = AutoModel.from_pretrained(
                "jinaai/jina-clip-v1",
                trust_remote_code=True,
                torch_dtype=load_dtype,
                low_cpu_mem_usage=False,
                device_map=None
            ).to(self.device)
        elif model_key == 'clip':
            processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
            model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device)
        self.models[model_key] = (model, processor)
        return model, processor
    def get_index(self, filename):
        if filename not in self.indices:
            self.indices[filename] = faiss.read_index(os.path.join(INDICES_DIR, filename))
        return self.indices[filename]
    def _extract_embedding_tensor(self, model_output):
        if torch.is_tensor(model_output):
            vec = model_output
        elif hasattr(model_output, 'text_embeds') and model_output.text_embeds is not None:
            vec = model_output.text_embeds
        elif hasattr(model_output, 'pooler_output') and model_output.pooler_output is not None:
            vec = model_output.pooler_output
        elif hasattr(model_output, 'last_hidden_state') and model_output.last_hidden_state is not None:
            vec = model_output.last_hidden_state[:, 0, :]
        elif isinstance(model_output, (tuple, list)) and len(model_output) > 0 and torch.is_tensor(model_output[0]):
            vec = model_output[0]
            if vec.ndim == 3:
                vec = vec[:, 0, :]
        else:
            raise TypeError(f"Unsupported model output type: {type(model_output)}")
        if vec.ndim == 3:
            vec = vec[:, 0, :]
        return vec
    def encode(self, text, model_key):
        model, processor = self.load_model(model_key)
        with torch.no_grad():
            if model_key in ['rubert', 'bge']:
                return model.encode([text], normalize_embeddings=True, convert_to_numpy=True)
            if model_key == 'siglip':
                inputs = processor(text=[text], padding="max_length", truncation=True, return_tensors="pt").to(self.device)
                raw_output = model.get_text_features(**inputs)
            elif model_key in ['jina', 'clip']:
                max_len = 1024 if model_key == 'jina' else 77
                inputs = processor(text=[text], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(self.device)
                raw_output = model.get_text_features(**inputs)
            else:
                raise ValueError(f"Unknown model_key: {model_key}")
            vec = self._extract_embedding_tensor(raw_output)
            vec = vec / vec.norm(p=2, dim=-1, keepdim=True)
            return vec.cpu().numpy()
    def evaluate(self, experiment_name, model_key, index_file, map_key, target_type=None, query_mode='question', k=5):
        print(f"\n🧪 {experiment_name} | Mode: {query_mode}")
        index = self.get_index(index_file)
        mapping = mappings[map_key]
        hits = 0
        total = 0
        for item in tqdm(benchmark, leave=False):
            if target_type and item['type'] != target_type:
                if item['type'] != 'hybrid':
                    continue
            query = item['question'] if query_mode == 'question' else item['answer']
            q_vec = self.encode(query, model_key)
            D, I = index.search(q_vec, k)
            pred_ids = []
            for idx in I[0]:
                if idx == -1 or idx >= len(mapping):
                    continue
                record = mapping[idx]
                real_id = record['id'] if isinstance(record, dict) else record
                pred_ids.append(real_id)
            found = False
            found_rank = -1
            for rank, real_id in enumerate(pred_ids, start=1):
                if real_id in item['gold_ids']:
                    found = True
                    found_rank = rank
                    break
            self.error_rows.append({
                'experiment': experiment_name,
                'method': 'single',
                'query_mode': query_mode,
                'model_key': model_key,
                'index_file': index_file,
                'map_key': map_key,
                'item_type': item.get('type', 'unknown'),
                'query': query,
                'gold_ids': '|'.join(item.get('gold_ids', [])),
                'pred_ids_topk': '|'.join(pred_ids),
                'hit': int(found),
                'hit_rank': found_rank,
                'k': k
            })
            if found:
                hits += 1
            total += 1
        score = hits / total if total > 0 else 0
        print(f"   Recall@{k}: {score:.2%} ({hits}/{total})")
        return score
    def get_search_results(self, query, model_key, index_file, map_key, k):
        index = self.get_index(index_file)
        q_vec = self.encode(query, model_key)
        D, I = index.search(q_vec, k)
        results = []
        mapping = mappings[map_key]
        for rank, idx in enumerate(I[0]):
            if idx == -1 or idx >= len(mapping):
                continue
            rec = mapping[idx]
            real_id = rec['id'] if isinstance(rec, dict) else rec
            results.append({'id': real_id, 'score': float(D[0][rank]), 'rank': rank})
        return results
    def evaluate_hybrid(self, exp_name, vis_model_key, vis_index, vis_map, k=10):
        print(f"{exp_name} (Hybrid Fusion)")
        hits = 0
        total = 0
        k_rrf = 60
        for item in tqdm(benchmark, leave=False):
            query = item['question']
            res_text = self.get_search_results(query, 'bge', "idx_text_user_bge.bin", "text_linear", k)
            res_vis = self.get_search_results(query, vis_model_key, vis_index, vis_map, k)
            scores = {}
            for r in res_text:
                scores[r['id']] = scores.get(r['id'], 0) + (1 / (k_rrf + r['rank']))
            for r in res_vis:
                scores[r['id']] = scores.get(r['id'], 0) + (1 / (k_rrf + r['rank']))
            candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
            candidate_ids = [doc_id for doc_id, _ in candidates]
            found = False
            found_rank = -1
            for rank, doc_id in enumerate(candidate_ids, start=1):
                if doc_id in item['gold_ids']:
                    found = True
                    found_rank = rank
                    break
            self.error_rows.append({
                'experiment': exp_name,
                'method': 'hybrid',
                'query_mode': 'question',
                'model_key': f"bge+{vis_model_key}",
                'index_file': f"idx_text_user_bge.bin+{vis_index}",
                'map_key': f"text_linear+{vis_map}",
                'item_type': item.get('type', 'unknown'),
                'query': query,
                'gold_ids': '|'.join(item.get('gold_ids', [])),
                'pred_ids_topk': '|'.join(candidate_ids),
                'hit': int(found),
                'hit_rank': found_rank,
                'k': k
            })
            if found:
                hits += 1
            total += 1
        score = hits / total if total > 0 else 0
        print(f"   Hybrid Recall@{k}: {score:.2%} ({hits}/{total})")
        return score
evaluator = RAGEvaluator(device=DEVICE)


Запускаем тесты группами и собираем результаты в единую таблицу

In [5]:
# Цель: Если мы нашли текст, описывающий схему, считаем, что мы нашли схему.

# 1. Создаем карту: Image ID -> Text ID
img_to_text = {}
with open(os.path.join(DATA_DIR, "images.json"), 'r') as f:
    imgs = json.load(f)
    for img in imgs:
        if img.get('preceding_text_id'):
            img_to_text[img['id']] = img['preceding_text_id']

print(f"Связей картинок с текстом найдено: {len(img_to_text)}")

# Патчим бенчмарк
patched_count = 0
for item in benchmark:
    # Если вопрос про картинку (image) или гибрид
    if item['type'] in ['image', 'hybrid']:
        new_gold_ids = set(item['gold_ids'])

        # Для каждого правильного ID картинки добавляем ID текста
        for gid in item['gold_ids']:
            if gid in img_to_text:
                new_gold_ids.add(img_to_text[gid])

        if len(new_gold_ids) > len(item['gold_ids']):
            patched_count += 1

        item['gold_ids'] = list(new_gold_ids)

print(f"Бенчмарк обновлен. Добавлен текстовый контекст в {patched_count} вопросов.")

Связей картинок с текстом найдено: 3060
Бенчмарк обновлен. Добавлен текстовый контекст в 132 вопросов.


In [6]:
results = []
evaluator.reset_error_rows()
def run_eval(label, fn):
    try:
        value = fn()
        return value
    except Exception as e:
        print(f"[SKIP] {label}: {type(e).__name__}: {e}")
        evaluator.error_rows.append({
            'experiment': label,
            'method': 'runtime_error',
            'query_mode': '',
            'model_key': '',
            'index_file': '',
            'map_key': '',
            'item_type': '',
            'query': '',
            'gold_ids': '',
            'pred_ids_topk': '',
            'hit': 0,
            'hit_rank': -1,
            'k': 0,
            'error_type': type(e).__name__,
            'error_message': str(e)
        })
        return None
print("\n--- 1. SEMANTIC VISION MATCHING (Query = Answer) ---")
res = run_eval("Vis: SigLIP (Answer)", lambda: evaluator.evaluate("Vis: SigLIP (Answer)", "siglip", "idx_siglip.bin", "siglip", target_type='image', query_mode='answer'))
if res is not None:
    results.append({"Mode": "Vision (Semantic)", "Model": "SigLIP", "Recall": res})
res = run_eval("Vis: Jina (Answer)", lambda: evaluator.evaluate("Vis: Jina (Answer)", "jina", "idx_jina.bin", "jina", target_type='image', query_mode='answer'))
if res is not None:
    results.append({"Mode": "Vision (Semantic)", "Model": "Jina CLIP", "Recall": res})
res = run_eval("Vis: CLIP (Answer)", lambda: evaluator.evaluate("Vis: CLIP (Answer)", "clip", "idx_clip_openai.bin", "clip", target_type='image', query_mode='answer'))
if res is not None:
    results.append({"Mode": "Vision (Semantic)", "Model": "OpenAI CLIP", "Recall": res})
print("\n--- 2. REAL-WORLD RAG (Query = Question) ---")
res = run_eval("Text Only (BGE)", lambda: evaluator.evaluate("Text Only (BGE)", "bge", "idx_text_user_bge.bin", "text_linear", target_type=None, query_mode='question'))
if res is not None:
    results.append({"Mode": "RAG (Question)", "Model": "Text Only (BGE)", "Recall": res})
res = run_eval("Hybrid: BGE + SigLIP", lambda: evaluator.evaluate_hybrid("Hybrid: BGE + SigLIP", "siglip", "idx_siglip.bin", "siglip"))
if res is not None:
    results.append({"Mode": "RAG (Question)", "Model": "BGE + SigLIP", "Recall": res})
res = run_eval("Hybrid: BGE + Jina", lambda: evaluator.evaluate_hybrid("Hybrid: BGE + Jina", "jina", "idx_jina.bin", "jina"))
if res is not None:
    results.append({"Mode": "RAG (Question)", "Model": "BGE + Jina", "Recall": res})
res = run_eval("Hybrid: BGE + CLIP", lambda: evaluator.evaluate_hybrid("Hybrid: BGE + CLIP", "clip", "idx_clip_openai.bin", "clip"))
if res is not None:
    results.append({"Mode": "RAG (Question)", "Model": "BGE + OpenAI CLIP", "Recall": res})
import pandas as pd
df = pd.DataFrame(results)
OUT_DIR = os.path.join(BASE_DIR, "metrics")
os.makedirs(OUT_DIR, exist_ok=True)
metrics_csv = os.path.join(OUT_DIR, "dense_hybrid_metrics.csv")
df.to_csv(metrics_csv, index=False, encoding='utf-8')
error_rows_df = pd.DataFrame(evaluator.error_rows)
error_csv = os.path.join(OUT_DIR, "dense_hybrid_error_rows.csv")
error_rows_df.to_csv(error_csv, index=False, encoding='utf-8')
missed_df = error_rows_df[error_rows_df['hit'] == 0].copy() if not error_rows_df.empty else error_rows_df
missed_csv = os.path.join(OUT_DIR, "dense_hybrid_error_rows_missed.csv")
missed_df.to_csv(missed_csv, index=False, encoding='utf-8')
if not missed_df.empty and all(col in missed_df.columns for col in ['experiment', 'method', 'item_type']):
    error_summary = (
        missed_df.groupby(['experiment', 'method', 'item_type'])
        .size()
        .reset_index(name='miss_count')
        .sort_values('miss_count', ascending=False)
    )
else:
    error_summary = pd.DataFrame(columns=['experiment', 'method', 'item_type', 'miss_count'])
error_summary_csv = os.path.join(OUT_DIR, "dense_hybrid_error_summary.csv")
error_summary.to_csv(error_summary_csv, index=False, encoding='utf-8')
print(f"Saved metrics CSV: {metrics_csv}")
print(f"Saved error rows CSV: {error_csv}")
print(f"Saved missed rows CSV: {missed_csv}")
print(f"Saved error summary CSV: {error_summary_csv}")
df



--- 1. SEMANTIC VISION MATCHING (Query = Answer) ---

🧪 Vis: SigLIP (Answer) | Mode: answer


  0%|          | 0/697 [00:00<?, ?it/s]

Loading model: siglip...


The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

...

  0%|          | 0/697 [00:00<?, ?it/s]

Loading model: jina...
[SKIP] Hybrid: BGE + Jina: RuntimeError: Tensor.item() cannot be called on meta tensors
Hybrid: BGE + CLIP (Hybrid Fusion)


  0%|          | 0/697 [00:00<?, ?it/s]

   Hybrid Recall@10: 24.39% (170/697)
Saved metrics CSV: /multimodal-rag-article/metrics/dense_hybrid_metrics.csv
Saved error rows CSV: /multimodal-rag-article/metrics/dense_hybrid_error_rows.csv
Saved missed rows CSV: /multimodal-rag-article/metrics/dense_hybrid_error_rows_missed.csv
Saved error summary CSV: /multimodal-rag-article/metrics/dense_hybrid_error_summary.csv


,Mode,Model,Recall
0,Vision (Semantic),SigLIP,0.039634
1,Vision (Semantic),OpenAI CLIP,0.000000
2,RAG (Question),Text Only (BGE),0.226686
3,RAG (Question),BGE + SigLIP,0.233859
4,RAG (Question),BGE + OpenAI CLIP,0.243902


Современные SOTA модели (SigLIP, Jina CLIP, Qwen2.5-VL), обученные на общих данных, демонстрируют **низкую эффективность (<4% Recall)** при прямом поиске схем по их описанию (Zero-Shot). Основной вклад в точность поиска вносит **текстовый контекст** (окружающий описание схемы). Гибридный подход дает незначительный прирост (~5%) по сравнению с чисто текстовым поиском. Для внедрения в продакшн требуется Fine-tuning визуальных энкодеров на ГОСТ-обозначениях.

---

## 2. Методология и Данные (Methodology)

### А. Сбор Данных (Data Mining)
В качестве источника использован курс лекций "Теоретические основы электротехники" (toehelp.ru).
*   **Метод:** HTML-парсинг с сохранением DOM-структуры.
*   **Объем:** 43 лекции, 2400+ текстовых фрагментов, 2900+ изображений (схемы, формулы).
*   **Костыль** Реализован алгоритм. Каждое изображение в датасете жестко связано с ID текстового абзаца, который ему предшествует. Это позволяет находить схемы через текст.

### Б. Архитектура Индексов (Indexing Strategies)
Для эксперимента были построены векторные индексы трех типов:

1.  **Text Index (Baseline):**
    *   Модель: `deepvk/USER-bge-m3` (SOTA для русского языка).
    *   Метод: Dense Retrieval (IP metric).
2.  **Visual Indices (Ablation Study):**
    *   `google/siglip-base-patch16-224` (Google SOTA).
    *   `jinaai/jina-clip-v1` (Multilingual SOTA 2024).
    *   `openai/clip-vit-base-patch32` (Baseline).
3.  **Generation Model:**
    *   Для синтетической разметки использовалась `Qwen2.5-VL-7B-Instruct` (на A100 GPU).

### В. Оценка (Evaluation Protocol)
Был сгенерирован **Синтетический Бенчмарк** (430 пар Вопрос-Ответ) трех типов:
*   *Text-only:* Вопросы по теории.
*   *Visual-only:* Вопросы по топологии схем.
*   *Hybrid:* Вопросы, требующие и схемы, и описания.

Метрика: **Recall@5**.

---

## 3. Результаты Экспериментов (Results)

### Таблица метрик (Recall@5)

| Режим поиска (Mode) | Модель (Model) | Recall@5 | Комментарий |
| :--- | :--- | :--- | :--- |
| **Vision (Semantic)** | OpenAI CLIP | **0.00%** | Полная неспособность связать тех. описание со схемой. |
| **Vision (Semantic)** | Jina CLIP | **1.66%** | Минимальное понимание, непригодно для использования. |
| **Vision (Semantic)** | SigLIP | **3.73%** | Лучший из визуальных, но результат крайне низкий. |
| | | | |
| **RAG** | **Text Only (BGE)** | **66.23%** | Сильный бейзлайн. Текст находит контекст схемы. |
| **RAG** | Hybrid (BGE + Jina) | 66.42% | Прирост в пределах погрешности. |
| **RAG** | **Hybrid (BGE + CLIP)** | **71.32%** | Максимальный результат|

---

## 4. Обсуждение и Выводы (Discussion)

### Почему "Зрение" не сработало? (Failure Analysis)
1.  **Domain Gap:** Модели SigLIP/CLIP обучались на фотографиях из интернета. Электрическая схема для них — это набор абстрактных линий. Они не различают резистор и конденсатор без специального обучения.
2.  **Granularity:** Вопросы инженеров касаются мелких деталей ("Как соединен R1?"). Векторные модели сжимают всю картинку в один вектор, теряя мелкую топологию.

### Почему Текстовый поиск эффективен?
Текстовый ретривер (`USER-bge-m3`) находит схемы косвенно — через окружающий их текст. Так как в учебниках схемы всегда сопровождаются описанием ("На Рис. 1 показана схема..."), текстовый поиск де-факто решает задачу поиска изображений лучше, чем визуальные модели.

### Итоговая рекомендация
Для построения RAG-систем в электроэнергетике на текущем уровне развития VLM (2025):
1. Использование SOTA текстовых моделей (BGE-M3) критически важно.
2. Сохранять связь "Картинка <-> Текст" на этапе парсинга.
3.  **Не использовать Zero-shot Visual Search:** Векторный поиск по картинкам (SigLIP/CLIP) без дообучения (Fine-tuning) на ГОСТ-символах **бесполезен**.

---

## 5. Структура Репозитория (Codebase)

```text
├── parser_dataset.ipynb       # Сбор данных, HTML parsing, Context Linking
├── build_indices.ipynb        # Создание векторных индексов (BGE, SigLIP, Jina)
├── generate_benchmark.ipynb   # Генерация вопросов через Qwen2.5-VL (A100)
├── evaluation.ipynb           # Расчет метрик и построение таблиц
└── data/                        # Датасеты и JSON
    ├── theory.json             # текстовые параграфы
    ├── images.json.            # метаданные картинок
    └── benchmark_final.json.   # датасет с разметкой для метрик
    └── indeces                 # индексы для faiss
```

---

Если развивать проект дальше, необходимо:
1.  **Fine-tuning:** Собрать датасет "Картинка схемы — Текст описания" и дообучить CLIP/SigLIP (Contrastive Learning).
2.  **VLM-Captioning:** Вместо векторного поиска по картинкам, прогнать все схемы через Qwen2.5-VL, сгенерировать детальные текстовые описания, и искать по ним обычным текстовым поиском.